# Trilha sonora — Niro Character Manager

Gera as faixas da trilha do slideshow com o MusicGen.

**Antes de rodar:** menu *Ambiente de execução → Alterar o tipo de ambiente de
execução → **T4 GPU** → Salvar*.

Rode as células **na ordem**. Se a sessão cair, rode de novo a 1 e a 2 antes de
voltar para as de geração — as faixas já prontas são puladas automaticamente.

⚠️ **Os arquivos somem quando a sessão do Colab encerra.** Baixe o zip (última
célula) antes de fechar.

## 1. Conferir a GPU

In [ ]:
!nvidia-smi

## 2. Carregar o modelo

Demora 2-3 minutos na primeira vez.

Para mais qualidade (e ~3x mais lento), troque `musicgen-small` por `facebook/musicgen-medium`.

In [ ]:
from transformers import pipeline
import torch

gerador = pipeline(
    "text-to-audio",
    model="facebook/musicgen-small",
    device=0 if torch.cuda.is_available() else -1,
)
print("Modelo carregado.")

## 3. Teste rápido (opcional)

Gera uma faixa só, para conferir se está tudo funcionando.

In [ ]:
import numpy as np
import scipy.io.wavfile
from IPython.display import Audio

prompt = "joyful fairy instrumental, celesta and glockenspiel, pizzicato strings, light flute, bright and playful, storybook magic, 115 BPM, tambourine, no vocals"

saida = gerador(prompt, forward_params={"do_sample": True, "max_new_tokens": 1500})

audio = np.squeeze(saida["audio"])
if audio.ndim > 1:
    audio = audio.T

scipy.io.wavfile.write("teste.wav", rate=saida["sampling_rate"], data=audio)
Audio("teste.wav")

## 4. Gerar os 12 elementos (36 faixas, ~25 a 35 min)

In [ ]:
TEMAS = {
    "aero_1":     "gentle wind instrumental, solo wooden flute and soft human whistling, airy sustained pads, distant wind chimes, calm and open, breeze over empty plains, 90 BPM, minimal percussion, no vocals",
    "aero_2":     "soaring adventure instrumental, pan flute lead over sweeping strings, bright horns, weightless and free, flight above the clouds, 110 BPM, light cymbal swells, no vocals",
    "aero_3":     "turbulent orchestral instrumental, rapid piccolo and flute runs, whistling wind textures, urgent string ostinato, swirling and violent gale, 145 BPM, driving percussion, no vocals",
    "aqua_1":     "flowing orchestral instrumental, legato strings and harp arpeggios, soft muted trumpet melody, gentle marimba, fluid and weightless, clear running river, 100 BPM, no vocals",
    "aqua_2":     "mysterious underwater instrumental, distant french horns, submerged string pads, sparse piano drops, deep reverb, unknown depths, 75 BPM, no drums, no vocals",
    "aqua_3":     "crushing deep sea instrumental, heavy low brass swells, dense cello section, slow taiko pulse, immense pressure, dark and vast, 65 BPM, no vocals",
    "bio_1":      "ancient nature instrumental, solo cello over native wooden flute, frame drum and seed rattles, warm and living, old runes in a deep forest, 80 BPM, no vocals",
    "bio_2":      "mystical growth instrumental, layered cellos, ocarina and pan pipes, earthy hand percussion, soft wordless humming, benevolent mystery, 95 BPM",
    "bio_3":      "ritual of life instrumental, tribal drums and shakers, cello ostinato, bone flute, layered chanting texture, primal and sacred, 110 BPM",
    "electro_1":  "electric hybrid orchestral instrumental, staccato strings with synth arpeggio, crackling energy textures, brass stabs, fast and charged, 130 BPM, no vocals",
    "electro_2":  "high energy rock hybrid instrumental, distorted electric guitar riff, driving live drums, synth bass pulse, adrenaline surge, 150 BPM, no vocals",
    "electro_3":  "static charge ambient instrumental, humming electrical drone, glitchy percussive clicks, slow detuned synth pad, restrained tension before the strike, 70 BPM, no vocals",
    "fae_1":      "joyful fairy instrumental, celesta and glockenspiel, pizzicato strings, light flute, bright and playful, storybook magic, 115 BPM, tambourine, no vocals",
    "fae_2":      "wonder and discovery instrumental, harp arpeggios, shimmering bells, warm strings swelling in awe, magic revealed, 90 BPM, no vocals",
    "fae_3":      "festive fairy dance instrumental, fiddle and tin whistle, hand claps and bodhran, accordion, communal celebration, 130 BPM, no vocals",
    "flama_1":    "heroic orchestral instrumental, triumphant french horns and trumpets, soaring strings, timpani and cymbals, brave and blazing, 120 BPM, no vocals",
    "flama_2":    "dangerous fire instrumental, low brass growls, aggressive string ostinato, crackling ember percussion, menacing and unpredictable, 135 BPM, no vocals",
    "flama_3":    "warm fireside instrumental, acoustic guitar and fiddle, hand percussion, bright horns, joyful and energetic, 110 BPM, no vocals",
    "glacial_1":  "extreme range instrumental, very high sustained violin harmonics over very low contrabass drone, glassy bells, cold and vast, frozen beauty, 60 BPM, no vocals",
    "glacial_2":  "heavy glacial instrumental, deep booming drums, low brass, piercing high string swells, crushing ice, dangerous and slow, 70 BPM, no vocals",
    "glacial_3":  "crystalline ambient instrumental, high celesta and glass harmonica, subsonic drone, sparse piano, beautiful and lethal stillness, 55 BPM, no percussion, no vocals",
    "kinetic_1":  "determined orchestral instrumental, driving string ostinato, steady taiko and snare, rising brass, relentless forward motion, 125 BPM, no vocals",
    "kinetic_2":  "athletic percussive instrumental, body percussion and stomping drums, punchy brass hits, building raw energy, human effort, 140 BPM, no vocals",
    "kinetic_3":  "resilient emotional instrumental, solo piano over swelling strings, slow building drums, struggle turning into resolve, 100 BPM, no vocals",
    "lumen_1":    "sacred instrumental, solo harp and warm string pad, wordless female choir, cathedral reverb, serene and divine, 70 BPM, no drums",
    "lumen_2":    "radiant sacred instrumental, full choir on open vowels, pipe organ and brass chorale, ringing bells, glorious and uplifting, 85 BPM",
    "lumen_3":    "quiet devotion instrumental, harp arpeggios, soft strings, distant boy soprano texture, intimate and reverent, 65 BPM, minimal, no drums",
    "mineral_1":  "grounded orchestral instrumental, low strings and heavy anvil percussion, steady tuba and trombone, unshakable and firm, 90 BPM, no vocals",
    "mineral_2":  "forge instrumental, metallic hammer percussion, industrial textures, deep brass, relentless and solid, 105 BPM, no vocals",
    "mineral_3":  "mountain ambient instrumental, deep earth drone, sparse low piano, distant stone percussion, ancient and immovable, 60 BPM, no vocals",
    "psy_1":      "meditative ambient instrumental, singing bowls and soft drone, slow breathing pads, tranquil and trance like, 60 BPM, no percussion, no vocals",
    "psy_2":      "psychic mystery instrumental, detuned celesta, reversed textures, whispering wordless voices, dissonant swells, uncanny and disorienting, 80 BPM",
    "psy_3":      "cosmic mind instrumental, warm analog pads, slow evolving drone, distant wordless choir, vast inner space, 65 BPM, no percussion",
    "umbra_1":    "sinister ritual instrumental, dark low male chanting texture, pipe organ, deep drums, church ruins reverb, forbidden and ominous, 75 BPM",
    "umbra_2":    "villain theme instrumental, menacing low brass, dissonant string clusters, slow ticking percussion, cold and calculating, 85 BPM, no vocals",
    "umbra_3":    "creeping dread ambient instrumental, sub bass drone, scraping metallic textures, distant whispers, fear of the unseen, 55 BPM, no clear beat",
}

import os, numpy as np, scipy.io.wavfile

os.makedirs("trilhas", exist_ok=True)

for nome, p in TEMAS.items():
    # Retomada: se o mp3 já existe, pula. Apague o arquivo para refazer a faixa.
    if os.path.exists(f"trilhas/{nome}.mp3"):
        print("Já existe, pulando:", nome)
        continue
    print("Gerando:", nome)
    saida = gerador(p, forward_params={"do_sample": True, "max_new_tokens": 1500})
    audio = np.squeeze(saida["audio"])
    if audio.ndim > 1:
        audio = audio.T
    scipy.io.wavfile.write(f"trilhas/{nome}.wav", rate=saida["sampling_rate"], data=audio)
    os.system(
        f"ffmpeg -y -loglevel error -i trilhas/{nome}.wav "
        f"-af loudnorm=I=-16:TP=-1.5:LRA=11 -b:a 160k trilhas/{nome}.mp3"
    )
    os.remove(f"trilhas/{nome}.wav")

print("Pronto. Arquivos em trilhas/")

## 5. Gerar as regiões (18 faixas, ~12 a 18 min)

**Renomeie as chaves** (`cidadela_1`, `gelo_1`…) para os nomes das suas regiões
antes de rodar, e ajuste o bioma e os instrumentos no texto de cada prompt.

In [ ]:
TEMAS = {
    "cidadela_1": "noble orchestral instrumental, heroic french horns, full string section, timpani and cymbal swells, proud and ceremonial, royal capital, 100 BPM, no vocals",
    "cidadela_2": "warm medieval city instrumental, lute and hurdy gurdy, tambourine, fiddle melody, bustling and welcoming, market square, 115 BPM, no vocals",
    "cidadela_3": "epic sacred orchestral instrumental, full latin style choir singing wordless vowels, cathedral reverb, brass chorale and pipe organ, solemn and monumental, 80 BPM",
    "gelo_1":     "cold ambient orchestral instrumental, sustained high strings, low drone, sparse piano notes, icy and desolate, frozen wasteland, 60 BPM, no percussion, no vocals",
    "gelo_2":     "somber marching instrumental, low male wordless chant, heavy drums, bowed cellos, cold and relentless, warriors crossing the ice, 95 BPM",
    "gelo_3":     "mournful sacred instrumental, distant latin style choir on sustained vowels, solo cello, deep drone, frozen cathedral atmosphere, grieving and vast, 65 BPM",
    "floresta_1": "mystical forest ambient instrumental, wooden flute, soft nylon guitar, nature textures, gentle strings, ancient and alive, 75 BPM, no vocals",
    "floresta_2": "tribal ritual instrumental, frame drums and shakers, low wooden flute, layered wordless chanting, primal and hypnotic, 105 BPM",
    "floresta_3": "sacred nature instrumental, soft latin style female choir on open vowels, harp and low strings, forest reverb, reverent and ancient, 70 BPM",
    "deserto_1":  "arid desert instrumental, duduk and oud, sparse frame drum, shimmering heat pads, lonely and endless, sun scorched dunes, 80 BPM, no vocals",
    "deserto_2":  "exotic rhythmic instrumental, darbuka and riq percussion, oud melody, low strings drone, traveling and determined, 120 BPM, no vocals",
    "deserto_3":  "ancient ruins instrumental, distant latin style choir on long vowels, low drone, sparse metallic percussion, haunting and forgotten, buried civilization, 70 BPM",
    "costa_1":    "coastal folk instrumental, acoustic guitar and accordion, soft fiddle, gentle wave textures, salty and welcoming, harbor at sunset, 95 BPM, no vocals",
    "costa_2":    "adventurous sea instrumental, sweeping strings, bold brass, rolling snare, hopeful and expansive, ship leaving port, 125 BPM, no vocals",
    "costa_3":    "melancholic maritime instrumental, low male latin style choir on sustained vowels, creaking ship textures, solo violin, deep water reverb, mournful, 70 BPM",
    "sombrias_1": "dark ominous instrumental, low brass drones, dissonant string clusters, sparse deep drums, oppressive and dreadful, cursed land, 60 BPM, no vocals",
    "sombrias_2": "dark orchestral action instrumental, aggressive low strings ostinato, pounding taiko, brass hits, urgent and threatening, 140 BPM, no vocals",
    "sombrias_3": "sinister sacred instrumental, dark latin style male choir chanting on low vowels, pipe organ, deep drums, church ruins reverb, ominous and ritualistic, 75 BPM",
}

import os, numpy as np, scipy.io.wavfile

os.makedirs("trilhas", exist_ok=True)

for nome, p in TEMAS.items():
    # Retomada: se o mp3 já existe, pula. Apague o arquivo para refazer a faixa.
    if os.path.exists(f"trilhas/{nome}.mp3"):
        print("Já existe, pulando:", nome)
        continue
    print("Gerando:", nome)
    saida = gerador(p, forward_params={"do_sample": True, "max_new_tokens": 1500})
    audio = np.squeeze(saida["audio"])
    if audio.ndim > 1:
        audio = audio.T
    scipy.io.wavfile.write(f"trilhas/{nome}.wav", rate=saida["sampling_rate"], data=audio)
    os.system(
        f"ffmpeg -y -loglevel error -i trilhas/{nome}.wav "
        f"-af loudnorm=I=-16:TP=-1.5:LRA=11 -b:a 160k trilhas/{nome}.mp3"
    )
    os.remove(f"trilhas/{nome}.wav")

print("Pronto. Arquivos em trilhas/")

## 6. Baixar tudo

In [ ]:
!zip -qr trilhas.zip trilhas
from google.colab import files
files.download("trilhas.zip")

## Refazer uma faixa específica

A célula de geração pula o que já existe. Para refazer só a `umbra_2`, por
exemplo, apague o arquivo e rode a célula de novo:

```python
import os
os.remove("trilhas/umbra_2.mp3")
```

## Faixas mais longas que 30 segundos

`max_new_tokens` controla a duração — o MusicGen gera 50 tokens por segundo de
áudio:

| `max_new_tokens` | Duração |
|---|---|
| 1500 | ~30s |
| 3000 | ~1 min |
| 4500 | ~1min30 |
| 6000 | ~2 min |

**O modelo foi treinado com trechos de 30 segundos.** Além disso ele continua
gerando, mas vai perdendo o rumo: repete demais, muda de ideia, às vezes desmancha
a melodia. Até ~1 minuto costuma segurar bem; acima de 2 minutos a chance de sair
algo estranho é alta. O tempo de geração também é proporcional — faixas de 1
minuto dobram o tempo do lote.

## Alternativa melhor: esticar por repetição

Instantâneo e sem perda de qualidade, já que a trilha toca em loop por baixo do
slideshow de qualquer forma:

```python
import os, glob

for caminho in glob.glob("trilhas/*.mp3"):
    if caminho.endswith("_longo.mp3"):
        continue
    destino = caminho.replace(".mp3", "_longo.mp3")
    os.system(f"ffmpeg -y -loglevel error -stream_loop 5 -i {caminho} -b:a 160k {destino}")

print("Pronto.")
```

`-stream_loop 5` repete 6 vezes: 30s viram 3 minutos.

> Como há **3 faixas por elemento**, o player do slideshow pode alternar entre
> elas em vez de repetir a mesma — 30s × 3 já dão 1min30 de variação real. Por
> isso o padrão de 1500 tokens costuma bastar.

